# Thompson Sampling from Scratch — Build It in 10 Lines

**Goal:** Implement Thompson Sampling for a Bernoulli bandit and watch it learn which commodity trading strategy works best.

**Time:** 15 minutes

**What you'll learn:**
- How Thompson Sampling works with just Beta distributions and sampling
- Why posteriors naturally balance exploration and exploitation
- How beliefs evolve from uncertain to confident as data accumulates

## Quick Win: Thompson Sampling in 2 Minutes

Here's the complete algorithm — sample from posteriors, pick best sample, update beliefs.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta
import seaborn as sns

sns.set_style('whitegrid')
np.random.seed(42)

# 4 commodity trading strategies with unknown win rates
true_probs = np.array([0.45, 0.60, 0.52, 0.48])
strategy_names = ['Momentum', 'Mean-Reversion', 'Carry', 'Breakout']

# Thompson Sampling: Maintain Beta posteriors
alpha = np.ones(4)  # Successes + 1 (prior)
beta_param = np.ones(4)  # Failures + 1 (prior)

# Run 100 rounds
for t in range(100):
    # Sample from each posterior
    samples = np.random.beta(alpha, beta_param)
    
    # Pick strategy with highest sample
    strategy = np.argmax(samples)
    
    # Observe outcome (1 = win, 0 = loss)
    reward = np.random.binomial(1, true_probs[strategy])
    
    # Update posterior
    if reward == 1:
        alpha[strategy] += 1
    else:
        beta_param[strategy] += 1

print("After 100 rounds:")
for i, name in enumerate(strategy_names):
    posterior_mean = alpha[i] / (alpha[i] + beta_param[i])
    print(f"{name:15s}: True={true_probs[i]:.2f}, Learned={posterior_mean:.2f}, Pulls={int(alpha[i] + beta_param[i] - 2)}")

best_strategy = np.argmax(alpha / (alpha + beta_param))
print(f"\nBest strategy identified: {strategy_names[best_strategy]}")

In [ ]:
learning_objectives(["Thompson Sampling = sample from posteriors, pick best sample, update beliefs", "Beta distributions naturally track win/loss counts via conjugacy", "Exploration happens automatically through posterior uncertainty", "No tuning parameters needed (unlike ε-greedy's ε)"])

In [ ]:
apply_course_theme()
apply_plot_theme()

**That's it!** Thompson Sampling in 15 lines. Notice:
- Mean-Reversion (0.60 true win rate) should get most pulls
- Posterior means converge toward true probabilities
- No tuning parameters needed

## How It Works: Visualizing Posterior Evolution

Let's run Thompson Sampling again and watch the Beta distributions evolve.

In [ ]:
def plot_posteriors(alpha, beta_param, strategy_names, true_probs, title):
    """Plot Beta posteriors for all strategies."""
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.flatten()
    
    x = np.linspace(0, 1, 200)
    
    for i, (ax, name) in enumerate(zip(axes, strategy_names)):
        y = beta.pdf(x, alpha[i], beta_param[i])
        ax.fill_between(x, y, alpha=0.3)
        ax.plot(x, y, linewidth=2, label=f'Beta({int(alpha[i])}, {int(beta_param[i])})')
        ax.axvline(true_probs[i], color='red', linestyle='--', linewidth=2, label=f'True: {true_probs[i]}')
        ax.axvline(alpha[i]/(alpha[i]+beta_param[i]), color='blue', linestyle=':', linewidth=2, label='Posterior Mean')
        ax.set_title(f'{name}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Win Rate')
        ax.set_ylabel('Density')
        ax.legend(fontsize=9)
        ax.set_xlim(0, 1)
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Run Thompson Sampling and snapshot posteriors at key rounds
alpha = np.ones(4)
beta_param = np.ones(4)
snapshots = [10, 100, 500, 2000]
snapshot_idx = 0

# Initial state
plot_posteriors(alpha, beta_param, strategy_names, true_probs, "Round 0: Uniform priors")

for t in range(1, 2001):
    samples = np.random.beta(alpha, beta_param)
    strategy = np.argmax(samples)
    reward = np.random.binomial(1, true_probs[strategy])
    
    if reward == 1:
        alpha[strategy] += 1
    else:
        beta_param[strategy] += 1
    
    # Plot at snapshot rounds
    if snapshot_idx < len(snapshots) and t == snapshots[snapshot_idx]:
        plot_posteriors(alpha, beta_param, strategy_names, true_probs, f"Round {t}")
        snapshot_idx += 1

**Observe:**
- **Round 10:** Wide distributions, high uncertainty, lots of exploration
- **Round 100:** Distributions narrowing, Mean-Reversion pulling ahead
- **Round 500:** Clear separation, posteriors concentrated
- **Round 2000:** Very tight around true values, minimal exploration

**Key insight:** Exploration happens automatically because uncertain posteriors produce diverse samples. As posteriors tighten, exploration naturally decreases.

## Performance: Cumulative Regret vs ε-Greedy

Let's compare Thompson Sampling to ε-greedy (ε=0.1) on the same problem.

In [ ]:
def run_thompson_sampling(true_probs, T=2000):
    """Run Thompson Sampling and return regret."""
    K = len(true_probs)
    alpha = np.ones(K)
    beta_param = np.ones(K)
    regret = np.zeros(T)
    optimal_reward = np.max(true_probs)
    
    for t in range(T):
        samples = np.random.beta(alpha, beta_param)
        arm = np.argmax(samples)
        reward = np.random.binomial(1, true_probs[arm])
        
        if reward == 1:
            alpha[arm] += 1
        else:
            beta_param[arm] += 1
        
        regret[t] = optimal_reward - true_probs[arm]
    
    return np.cumsum(regret)

def run_epsilon_greedy(true_probs, epsilon=0.1, T=2000):
    """Run ε-greedy and return regret."""
    K = len(true_probs)
    counts = np.zeros(K)
    values = np.zeros(K)
    regret = np.zeros(T)
    optimal_reward = np.max(true_probs)
    
    for t in range(T):
        if np.random.random() < epsilon:
            arm = np.random.randint(K)  # Explore
        else:
            arm = np.argmax(values)  # Exploit
        
        reward = np.random.binomial(1, true_probs[arm])
        counts[arm] += 1
        values[arm] += (reward - values[arm]) / counts[arm]
        regret[t] = optimal_reward - true_probs[arm]
    
    return np.cumsum(regret)

# Run both algorithms 20 times and average
n_runs = 20
ts_regrets = np.zeros((n_runs, 2000))
eg_regrets = np.zeros((n_runs, 2000))

for i in range(n_runs):
    ts_regrets[i] = run_thompson_sampling(true_probs)
    eg_regrets[i] = run_epsilon_greedy(true_probs)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(ts_regrets.mean(axis=0), label='Thompson Sampling', linewidth=2)
plt.fill_between(range(2000), 
                 ts_regrets.mean(axis=0) - ts_regrets.std(axis=0),
                 ts_regrets.mean(axis=0) + ts_regrets.std(axis=0),
                 alpha=0.2)
plt.plot(eg_regrets.mean(axis=0), label='ε-greedy (ε=0.1)', linewidth=2)
plt.fill_between(range(2000),
                 eg_regrets.mean(axis=0) - eg_regrets.std(axis=0),
                 eg_regrets.mean(axis=0) + eg_regrets.std(axis=0),
                 alpha=0.2)
plt.xlabel('Round', fontsize=12)
plt.ylabel('Cumulative Regret', fontsize=12)
plt.title('Thompson Sampling vs ε-Greedy\nCommodity Trading Strategy Selection', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final regret (avg over {n_runs} runs):")
print(f"Thompson Sampling: {ts_regrets.mean(axis=0)[-1]:.1f}")
print(f"ε-greedy:          {eg_regrets.mean(axis=0)[-1]:.1f}")

**Result:** Thompson Sampling typically achieves lower regret than ε-greedy because:
1. Exploration decays naturally (no fixed ε)
2. Exploration is directed toward uncertain arms, not uniform random
3. Posterior concentration guides exploitation without tuning

## Modify This: Experiment with Different Settings

Try these modifications to build intuition:

In [ ]:
# EXPERIMENT 1: Change true probabilities
# Make them very close together — harder problem
true_probs_hard = np.array([0.48, 0.50, 0.49, 0.48])

# How much longer does it take to identify the best?
# Try running Thompson Sampling with true_probs_hard

# EXPERIMENT 2: Different priors
# Optimistic prior: Beta(5, 1) — assume everything works well
# Pessimistic prior: Beta(1, 5) — assume everything fails
# How does this affect early exploration?

# EXPERIMENT 3: Strong prior that's WRONG
# Start with Beta(10, 10) for all arms (prior belief: 50% win rate)
# But true win rates are [0.3, 0.6, 0.4, 0.45]
# How many rounds to overcome the wrong prior?

# YOUR CODE HERE
# Uncomment and modify:

# alpha_optimistic = np.ones(4) * 5
# beta_optimistic = np.ones(4)
# # ... run Thompson Sampling with these priors ...

## Summary

**What you learned:**
1. Thompson Sampling = sample from posteriors, pick best sample, update beliefs
2. Beta distributions naturally track win/loss counts via conjugacy
3. Exploration happens automatically through posterior uncertainty
4. No tuning parameters needed (unlike ε-greedy's ε)

**Commodity trading context:**
- Each strategy is an arm (Momentum, Mean-Reversion, Carry, Breakout)
- Each trade is a Bernoulli trial (win/loss)
- Thompson Sampling adaptively allocates capital to winning strategies
- Posteriors quantify uncertainty → natural risk management

**Next steps:**
- `02_belief_evolution.ipynb` — Watch posteriors converge in detail
- `03_gaussian_thompson_commodities.ipynb` — Apply to real commodity returns (Gaussian rewards)

In [ ]:
key_takeaways(["1. Thompson Sampling = sample from posteriors, pick best sample, update beliefs", "- Each strategy is an arm (Momentum, Mean-Reversion, Carry, Breakout)", "Each trade is a Bernoulli trial (win/loss)", "Thompson Sampling adaptively allocates capital to winning strategies", "Posteriors quantify uncertainty → natural risk management", "- `02_belief_evolution.ipynb` — Watch posteriors converge in detail"])